# 🤖 Ultimate Quant v3
Laboratorio de investigación del bot. **El código ya no vive aquí**: la celda 1 descarga
`quant_core.py` y `backtest.py` desde GitHub, así Colab ejecuta exactamente lo mismo que el bot.

Para cambiar la estrategia, se edita el repo (no este notebook) y se vuelve a correr la celda 1.

| Celda | Qué hace |
|---|---|
| 1 | Instala librerías y descarga el código del repo |
| 2 | Escáner de hoy (igual que el mensaje de cierre) |
| 3 | Estadísticas de tus operaciones reales |
| 4 | Descarga 10 años de datos históricos (~2 min) |
| 5 | Backtest de la estrategia actual vs SPY |
| 6 | Walk-forward: ¿los parámetros funcionan en años no vistos? |
| 7 | ADN por activo con las reglas reales de salida |
| 8 | Validación de candidatos nuevos |
| 9 | Comparar umbrales de un parámetro |
| 10 | Bitácora de operaciones |
| 11 | Guardar resultados en Google Drive |

## Celda 1 — Instalar y descargar el código del repo

In [ ]:
!pip install -q yfinance tabulate
import os, sys, importlib

REPO = 'https://github.com/Galactic20/Quant.git'
DIR  = '/content/Quant'
if os.path.isdir(DIR):
    !git -C {DIR} pull -q
else:
    !git clone -q {REPO} {DIR}
%cd {DIR}
if DIR not in sys.path:
    sys.path.insert(0, DIR)

import quant_core as qc, backtest as bt
importlib.reload(qc); importlib.reload(bt)
import pandas as pd
from IPython.display import display, Markdown
pd.set_option('display.width', 200)

# Capital con el que se calculan los tamaños de posición
CAPITAL = qc.PARAMETROS['CAPITAL_INICIAL']
print(f'✅ Código del repo cargado | Universo: {len(qc.SECTORES)} activos | Capital: ${CAPITAL}')

## Celda 2 — 🔥 Escáner de hoy

In [ ]:
datos_hoy = qc.descargar_datos_globales(periodo='3y')
df_senales = qc.escanear_universo(datos_globales=datos_hoy)
qc.imprimir_reporte(df_senales)

## Celda 3 — 📒 Tus operaciones reales
Se leen de `datos/operaciones_reales.json` en el repo. Para registrar una nueva,
edita ese archivo en GitHub y vuelve a correr la celda 1.

In [ ]:
display(bt.resumen_operaciones_reales().T)
ops_reales = bt.cargar_operaciones_reales()
display(ops_reales[ops_reales['origen'] == 'bot'][['ticker', 'fecha_cierre', 'invertido', 'pnl', 'motivo']])

## Celda 4 — 📡 Datos históricos (10 años)

In [ ]:
datos, spy, vix = bt.descargar('10y')
INICIO = spy.index[0] + pd.DateOffset(days=300)   # tras calentar la SMA 200
print(f'Periodo de prueba: {INICIO.date()} → {spy.index[-1].date()}')

## Celda 5 — 📊 Backtest de la estrategia actual vs SPY
Simula día a día con las reglas del bot: entradas al cierre, Stop Loss y Take Profit,
Chandelier, SMA 200, límite por sector, fracciones de acción y costes (`bt.BT_DEFAULTS`).

In [ ]:
res   = bt.backtest(datos, spy, vix, capital=CAPITAL, inicio=INICIO)
bench = bt.benchmark(spy, CAPITAL, inicio=INICIO)
display(Markdown(bt.reporte(res, bench)))
bt.graficar(res, bench)

## Celda 6 — 🧪 Walk-forward
Para cada año: elige los mejores parámetros con los 3 años anteriores y los mide en ese año,
que no vio. Si `opt_` no supera a `spy_` de forma consistente, la estrategia no tiene ventaja.

In [ ]:
wf = bt.walk_forward(datos, spy, vix, capital=CAPITAL)
display(wf)

## Celda 7 — 🔬 ADN por activo (con salidas reales, no retorno a 10 días)

In [ ]:
adn = bt.adn_por_activo(datos, spy, vix, inicio=INICIO)
display(adn[adn['operaciones'] >= qc.PARAMETROS['MIN_SEÑALES_ADN']].head(25))

## Celda 8 — 🧪 Validación de candidatos nuevos
Mismas reglas del bot. Criterio: ≥ 5 operaciones, profit factor ≥ 1.3 y esperanza > 0.

In [ ]:
CANDIDATOS = {
    'LMT': 'Defensa',
    'CRWD': 'Ciberseguridad',
}
qc.SECTORES.update(CANDIDATOS)   # solo en esta sesión, para ubicar su ETF de sector
datos_c = qc.descargar_datos_globales(list(CANDIDATOS), periodo='10y')
val = bt.adn_por_activo(datos_c, spy, vix, tickers=list(CANDIDATOS), inicio=INICIO)
val['veredicto'] = val.apply(lambda r: '✅ APROBADO' if r['operaciones'] >= 5
                             and r.get('profit_factor', 0) >= 1.3
                             and r.get('esperanza_trade', 0) > 0 else '❌', axis=1)
display(val)

## Celda 9 — ⚖️ Comparar umbrales de un parámetro
Reemplaza la antigua comparación RSI < 40 vs < 50, que usaba un RSI distinto al del bot.

In [ ]:
PARAMETRO = 'RSI_ENTRADA'
VALORES   = [40, 45, 50]
filas = []
for v in VALORES:
    p = {**qc.PARAMETROS, PARAMETRO: v}
    m = bt.backtest(datos, spy, vix, p, capital=CAPITAL, inicio=INICIO)['metricas']
    filas.append({PARAMETRO: v, **{k: m.get(k) for k in
                  ('retorno_total_pct', 'max_drawdown_pct', 'sharpe', 'operaciones',
                   'win_rate_pct', 'profit_factor')}})
display(pd.DataFrame(filas))
print('⚠️  Elegir el mejor valor aquí es sobreajuste: confírmalo con la celda 6.')

## Celda 10 — 📒 Bitácora de operaciones

In [ ]:
journal = qc.BitacoraTrades()
# journal.registrar_compra(ticker='NVDA', precio=135.50, unidades=0.5,
#                          stop_loss=128.40, calidad='NORMAL', take_profit=150.0)
# journal.cerrar_operacion(id_op=1, precio_salida=155.00)
journal.revisar_pendientes()
# Nota: posiciones.json aquí es una copia del repo. Las posiciones que vigila
# el bot se editan en GitHub (posiciones.json).

## Celda 11 — 💾 Guardar resultados en Google Drive *(opcional)*

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
CARPETA = '/content/drive/MyDrive/QuantBot/'
os.makedirs(CARPETA, exist_ok=True)
if 'res' in dir():
    res['operaciones'].to_csv(CARPETA + 'backtest_operaciones.csv', index=False)
    print('✅ backtest_operaciones.csv')
if 'wf' in dir():
    wf.to_csv(CARPETA + 'walk_forward.csv', index=False)
    print('✅ walk_forward.csv')
if os.path.exists('mi_bitacora.json'):
    shutil.copy('mi_bitacora.json', CARPETA + 'mi_bitacora.json')
    print('✅ mi_bitacora.json')